# PathSnap 1 · Train and compare five classifiers

This notebook reruns the original Python training scripts. It fits five algorithms on 80 training trips with five trip-disjoint folds, then scores each final refit on ten student-test trips.

Use Python 3.13. The setup cell installs the training dependencies into the selected notebook kernel if any are missing. If you already ran a cell that imported NumPy, restart the kernel before running all cells.

In [ ]:
from importlib.util import find_spec
import subprocess
import sys

packages = {
    "numpy": "numpy==2.1.3",
    "scipy": "scipy==1.15.3",
    "sklearn": "scikit-learn==1.6.1",
    "joblib": "joblib==1.4.2",
    "threadpoolctl": "threadpoolctl==3.5.0",
}
if any(find_spec(name) is None for name in packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages.values()])
else:
    print("Training dependencies are installed in this kernel.")

In [ ]:
from pathlib import Path
import csv
import hashlib
import os
import subprocess
import sys
import tempfile
from IPython.display import Markdown, display

# Set these paths if the notebook or dataset is outside the extracted project.
PROJECT_ROOT = None
DATASET_ZIP = None
cwd = Path.cwd()
if PROJECT_ROOT is None:
    PROJECT_ROOT = next((p for p in (cwd, *cwd.parents)
                         if (p / "stage-4" / "compare.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Set PROJECT_ROOT above to the extracted PathSnap folder.")
root = Path(PROJECT_ROOT).expanduser().resolve()
archive = (Path(DATASET_ZIP).expanduser().resolve() if DATASET_ZIP is not None
           else root / "inputs" / "Student folder.zip")
if not archive.is_file():
    raise FileNotFoundError(f"Place the original student dataset ZIP here: {archive}")

work = root / "work" / "notebook-runs"
work.mkdir(parents=True, exist_ok=True)
print("Dataset:", archive)

def run_python(*args):
    command = [sys.executable, "-u", *map(str, args)]
    env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", LOKY_MAX_CPU_COUNT="3")
    with subprocess.Popen(command, cwd=root, env=env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                          errors="replace") as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        exit_code = process.wait()
    if exit_code:
        raise subprocess.CalledProcessError(exit_code, command)

def read_rows(file):
    with file.open(encoding="utf-8", newline="") as stream:
        return list(csv.DictReader(stream))

def show_table(headers, rows):
    lines = [headers, ["---"] * len(headers), *rows]
    display(Markdown("\n".join("| " + " | ".join(map(str, row)) + " |" for row in lines)))

## Build or reuse the candidate features

This uses the original preprocessing and scoring rules. Reference-derived labels are stored separately from GPS and road features.

In [ ]:
import numpy
import scipy
import sklearn
import joblib

print(f"NumPy {numpy.__version__}, SciPy {scipy.__version__}, "
      f"scikit-learn {sklearn.__version__}, joblib {joblib.__version__}")
stage3 = root / "stage-3"
data_hash = hashlib.sha256(archive.read_bytes()).hexdigest()[:12]
source_hash = hashlib.sha256(b"".join(
    (stage3 / name).read_bytes()
    for name in ("candidates.py", "geometry.py", "reference.py", "run_stage3.py")
)).hexdigest()[:12]
cache = work / f"cache-{data_hash}-{source_hash}"
marker = cache / "complete"
if not marker.is_file():
    cache.mkdir(parents=True, exist_ok=True)
    run_python(stage3 / "run_stage3.py", archive, "--out", cache / "baseline", "--cache", cache)
    marker.touch()
else:
    print("Reusing candidate features from this dataset and source revision.")
print("Candidate cache:", cache)

## Inspect the actual training configurations

The sixth experiment is an ablation of logistic regression with the heading features removed.

In [ ]:
run_python("-c", f'''
import inspect, sys
sys.path.insert(0, {str(root / "stage-4")!r})
from compare import ALGORITHMS, estimator
from candidates import FEATURES
print(inspect.getsource(estimator))
print("Features:", ", ".join(FEATURES))
print("Experiments:", ", ".join(ALGORITHMS))
''')

## Train, refit and verify

The outputs are written to a new run directory under the project’s work folder. The original experiment files stay available for comparison.

In [ ]:
comparison_dir = Path(tempfile.mkdtemp(prefix="model-comparison-", dir=work))
run_python(root / "stage-4" / "compare.py", archive, cache, "--out", comparison_dir)
run_python(root / "stage-4" / "verify.py", cache, "--out", comparison_dir)
print("Fresh results and model files:", comparison_dir)

## Accuracies from this run

Top-1 counts eligible GPS points matched to the derived reference arc. Macro F1 averages the trip-local scores with equal weight per trip.

In [ ]:
measured = {(r["algorithm"], r["split"]): r for r in read_rows(comparison_dir / "comparison.csv")}
names = {
    "logistic": "Logistic regression",
    "gaussian_nb": "Gaussian Naive Bayes",
    "tree": "Decision tree",
    "forest": "Random forest",
    "hist_boost": "Histogram gradient boosting",
    "logistic_no_heading": "Logistic regression without heading",
}
table = []
for name, label in names.items():
    test = measured[name, "student_test"]
    cv = measured[name, "cv"]
    table.append([label, f'{float(cv["macro_f1"]):.4f}', f'{float(test["accuracy"]):.2%}',
                  f'{float(test["macro_f1"]):.4f}', f'{int(test["correct"]):,} / {int(test["eligible"]):,}'])
show_table(["Model", "CV Macro F1", "Student-test Top-1", "Student-test Macro F1",
            "Correct / eligible"], table)
test = measured["hist_boost", "student_test"]
print(f"Point-label coverage: {int(test['eligible']) / int(test['gps']):.2%}")